# 🎬 High-End Anime Combat Production Pipeline (ComfyUI + AnimateDiff + ControlNet + 60FPS RIFE)\n### Project: Dynamic Battle Engine | Characters: Nightshade vs. Prisoner\n**Author:** mhommdkuyr/dynamic-battle-engine  \n**Hardware:** Google Colab Free Tier (Tesla T4 GPU, 15GB VRAM)  \n**Engines:** ComfyUI, AnimateDiff-Evolved, ControlNet (OpenPose/Depth), CogVideoX, Practical-RIFE, Real-ESRGAN

In [ ]:
#@title 1. Environment Setup & Clone ComfyUI (Free T4 GPU Optimized)\nimport os\nfrom google.colab import drive\n\n!nvidia-smi\n\n# Clone ComfyUI Core\n%cd /content\n!git clone https://github.com/comfyanonymous/ComfyUI.git\n%cd /content/ComfyUI\n!pip install -q xformers torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121\n!pip install -q -r requirements.txt\n\nprint(\"✅ ComfyUI Core Installed successfully!\")

In [ ]:
#@title 2. Install Anime & Animation Custom Nodes\n%cd /content/ComfyUI/custom_nodes\n\n# 1. AnimateDiff Evolved\n!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git\n\n# 2. ControlNet Aux (OpenPose & Depth)\n!git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git\n\n# 3. CogVideoX Wrapper\n!git clone https://github.com/kijai/ComfyUI-CogVideoXWrapper.git\n\n# 4. ComfyUI Frame Interpolation (RIFE 60fps)\n!git clone https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git\n\n# 5. ComfyUI Manager\n!git clone https://github.com/ltdrdata/ComfyUI-Manager.git\n\n# Install dependencies for custom nodes\n!pip install -q -r /content/ComfyUI/custom_nodes/comfyui_controlnet_aux/requirements.txt\n!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-Frame-Interpolation/requirements.txt\n\nprint(\"✅ Animation & Control Custom Nodes Installed!\")

In [ ]:
#@title 3. Download Anime Models, Motion Modules & ControlNet Weights\n\n# 1. AnimateDiff Motion Module (v3 SD1.5)\n!wget -c https://huggingface.co/guoyww/animatediff/resolve/main/v3_sd15_mm.ckpt -P /content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved/models/\n\n# 2. High-Quality Anime Checkpoint (Anything V5 / Animagine)\n!wget -c https://huggingface.co/ckpt/anything-v5.0/resolve/main/AnythingV5_PrtRE.safetensors -P /content/ComfyUI/models/checkpoints/\n\n# 3. ControlNet OpenPose & Depth SD1.5\n!wget -c https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_openpose.pth -P /content/ComfyUI/models/controlnet/\n!wget -c https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth -P /content/ComfyUI/models/controlnet/\n\n# 4. Real-ESRGAN Anime 4K Upscaler\n!wget -c https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth -P /content/ComfyUI/models/upscale_models/\n\nprint(\"✅ Checkpoints, Motion Modules, and Upscalers Ready!\")

In [ ]:
#@title 4. Launch ComfyUI with Cloudflare Tunnel (Instant Web UI)\nimport subprocess\nimport threading\nimport time\nimport socket\nimport urllib.request\n\n# Download cloudflared\n!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\n!dpkg -i cloudflared-linux-amd64.deb\n\ndef run_comfyui():\n    !python /content/ComfyUI/main.py --dont-print-server --listen 127.0.0.1 --port 8188 --fp16\n\nthreading.Thread(target=run_comfyui, daemon=True).start()\ntime.sleep(5)\n\n# Launch tunnel\nprint(\"🌐 Starting Cloudflare Public Tunnel for ComfyUI...\")\n!cloudflared tunnel --url http://127.0.0.1:8188

In [ ]:
#@title 5. Post-Processing Pipeline: 60 FPS RIFE Interpolation & 4K RealESRGAN\nimport cv2\nimport os\n\ndef interpolate_to_60fps(input_video_path, output_video_path):\n    \"\"\"\n    Calls Practical-RIFE to double or triple frames to reach a clean 60 FPS stream.\n    \"\"\"\n    print(f\"Processing {input_video_path} through Practical-RIFE to 60 FPS...\")\n    os.system(f\"python -m rfe.inference_video --video {input_video_path} --output {output_video_path} --fps 60 --scale 1.0\")\n    print(f\"Interpolation Complete: {output_video_path}\")\n\ndef upscale_to_4k(input_video_path, output_video_path):\n    \"\"\"\n    Passes frames through RealESRGAN_x4plus_anime_6B for pristine 4K lines.\n    \"\"\"\n    print(f\"Upscaling {input_video_path} to 4K Anime Quality...\")\n    os.system(f\"python -m realesrgan.inference_realesrgan -n RealESRGAN_x4plus_anime_6B -i {input_video_path} -o {output_video_path}\")\n    print(f\"4K Upscale Complete: {output_video_path}\")